In [ ]:
# !pip install -q torch_geometric
# !pip install -q class_resolver
# !pip3 install pymatting

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric
!pip install -q class_resolver
!pip3 install pymatting

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as nnFn
import torch.optim as optim
import numpy as np
import random
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch_geometric.data import Data
from torch_geometric.nn import ChebConv
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss
)
from torch.utils.data import TensorDataset, DataLoader, Subset
from torch_geometric.data import Data
from torch_geometric.nn import ARMAConv

/home/snu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np

feature_path = "/home/snu/Downloads/liver_dino_features.npy"

label_path = "/home/snu/Downloads/liver_dino_labels.npy"

features = np.load(
    feature_path
).astype(np.float32)

y_labels = np.load(
    label_path
).astype(np.int64)

print("Feature shape:", features.shape)

print("Label shape:", y_labels.shape)

print("\nClass distribution:")

unique, counts = np.unique(
    y_labels,
    return_counts=True
)

for cls, cnt in zip(unique, counts):

    print(f"Class {cls}: {cnt}")

Feature shape: (635, 768)
Label shape: (635,)

Class distribution:
Class 0: 200
Class 1: 435


In [15]:
import torch.nn as nn
import torch.nn.functional as nnFn
from torch_geometric.nn import GATConv

class GAT_Supervised(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        heads=2,
        activ="RELU",
        dropout=0.2
    ):
        super(GAT_Supervised, self).__init__()

        # -------- GAT layers --------
        self.gat1 = GATConv(
            input_dim, hidden_dim, heads=heads, concat=True, dropout=dropout
        )
        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)

        self.gat2 = GATConv(
            hidden_dim * heads, hidden_dim, heads=heads, concat=True, dropout=dropout
        )
        self.bn2 = nn.BatchNorm1d(hidden_dim * heads)

        self.gat3 = GATConv(
            hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout
        )
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        # -------- Classifier --------
        self.fc = nn.Linear(hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

        activations = {
            "SELU": nnFn.selu,
            "SiLU": nnFn.silu,
            "GELU": nnFn.gelu,
            "RELU": nnFn.relu,
            "ELU": nnFn.elu
        }
        self.act = activations.get(activ, nnFn.elu)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.gat1(x, edge_index)
        x = self.bn1(x)
        x = self.act(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index)
        x = self.bn2(x)
        x = self.act(x)
        x = self.dropout(x)

        x = self.gat3(x, edge_index)
        x = self.bn3(x)
        x = self.act(x)
        x = self.dropout(x)

        logits = self.fc(x)
        return logits


In [17]:
def create_adj(F, alpha=1):
    F_norm = F / np.linalg.norm(F, axis=1, keepdims=True)
    W = np.dot(F_norm, F_norm.T)
    W = np.where(W >= alpha, 1, 0).astype(np.float32)
    W = W / W.max()
    return W

def load_data(adj, node_feats):
    node_feats = torch.from_numpy(node_feats).float()
    edge_index = torch.from_numpy(np.array(np.nonzero((adj > 0))))
    return Data(x=node_feats, edge_index=edge_index)

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
alpha = 0.83
feats_dim = 768
hidden_dim = 256
num_classes = 2
num_epochs = 2000
lr = 0.0001
weight_decay = 1e-4
batch_print_freq = 100

In [19]:
W = create_adj(features, alpha)
data = load_data(W, features).to(device)
print(data)

Data(x=[635, 768], edge_index=[2, 65773])


In [20]:
sss = StratifiedShuffleSplit(n_splits=20, test_size=0.9, random_state=42)

accuracies, precisions, recalls, f1_scores, aucs, ce_losses = [], [], [], [], [], []

for fold, (train_idx, test_idx) in enumerate(sss.split(features, y_labels), start=1):
    print(f"\n=== Fold {fold} ===")

    train_idx_t = torch.from_numpy(train_idx).long().to(device)
    y_train_tensor = torch.from_numpy(y_labels[train_idx]).long().to(device)


    model = GAT_Supervised(feats_dim, hidden_dim, num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(data)
        loss = ce_loss(logits[train_idx_t], y_train_tensor)

        loss.backward()
        optimizer.step()

        if epoch % batch_print_freq == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                preds_train = logits[train_idx_t].argmax(dim=1)
                acc_train = accuracy_score(y_train_tensor.cpu(), preds_train.cpu())
            print(f"Fold {fold} Epoch {epoch}: Loss={loss.item():.6f} | TrainAcc={acc_train:.4f}")

    model.eval()
    with torch.no_grad():
        out = model(data)
        preds = out.argmax(dim=1).cpu().numpy()
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()

    y_test = y_labels[test_idx]
    y_pred_test = preds[test_idx]
    y_pred_test = preds[test_idx]
    y_prob_test = probs[test_idx]

    acc = accuracy_score(y_test, y_pred_test)
    prec = precision_score(y_test, y_pred_test, zero_division=0)
    rec = recall_score(y_test, y_pred_test, zero_division=0)
    f1 = f1_score(y_test, y_pred_test, zero_division=0)
    auc = roc_auc_score(y_test, y_prob_test)
    ce = log_loss(y_test, y_prob_test)

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)
    aucs.append(auc)
    ce_losses.append(ce)

    print(f"Fold {fold} → "
          f"Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | "
          f"F1={f1:.4f} | AUC={auc:.4f} | CE Loss={ce:.4f}")


# Final summary
print("\n=== Average Results Across 20 Folds ===")
print(f"Accuracy:  {np.mean(accuracies):.4f} \u00b1 {np.std(accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f} \u00b1 {np.std(precisions):.4f}")
print(f"Recall:    {np.mean(recalls):.4f} \u00b1 {np.std(recalls):.4f}")
print(f"F1-score:  {np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}")
print(f"AUC:       {np.mean(aucs):.4f} \u00b1 {np.std(aucs):.4f}")
print(f"CE Loss:   {np.mean(ce_losses):.4f} \u00b1 {np.std(ce_losses):.4f}")


=== Fold 1 ===
Fold 1 Epoch 1: Loss=0.700517 | TrainAcc=0.4762
Fold 1 Epoch 100: Loss=0.260336 | TrainAcc=0.9206
Fold 1 Epoch 200: Loss=0.130562 | TrainAcc=0.9524
Fold 1 Epoch 300: Loss=0.090319 | TrainAcc=0.9524
Fold 1 Epoch 400: Loss=0.062640 | TrainAcc=0.9841
Fold 1 Epoch 500: Loss=0.144493 | TrainAcc=0.9683
Fold 1 Epoch 600: Loss=0.044317 | TrainAcc=0.9841
Fold 1 Epoch 700: Loss=0.044908 | TrainAcc=0.9841
Fold 1 Epoch 800: Loss=0.046249 | TrainAcc=0.9841
Fold 1 Epoch 900: Loss=0.103145 | TrainAcc=0.9841
Fold 1 Epoch 1000: Loss=0.017569 | TrainAcc=1.0000
Fold 1 Epoch 1100: Loss=0.042655 | TrainAcc=0.9841
Fold 1 Epoch 1200: Loss=0.034303 | TrainAcc=0.9841
Fold 1 Epoch 1300: Loss=0.057175 | TrainAcc=0.9683
Fold 1 Epoch 1400: Loss=0.022584 | TrainAcc=1.0000
Fold 1 Epoch 1500: Loss=0.009409 | TrainAcc=1.0000
Fold 1 Epoch 1600: Loss=0.016462 | TrainAcc=1.0000
Fold 1 Epoch 1700: Loss=0.041263 | TrainAcc=0.9841
Fold 1 Epoch 1800: Loss=0.024909 | TrainAcc=1.0000
Fold 1 Epoch 1900: Loss=0.0